# Retrieval

We have 102 chunks stored in ChromaDB with embeddings from 1_load_chunk.

This notebook builds the retrieval layer:
- Take a plain text query
- Embed it using the same model
- Find the top-k most similar chunks using similarity search
- Inspect the results

This is the **R** in RAG — before any LLM is involved.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded:", "✅" if OPENAI_API_KEY else "❌ NOT FOUND")

## Load the vector store

In [ ]:
use_OpenAI = True

if use_OpenAI:
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    vectorstore = Chroma(
        persist_directory="../chroma_DB/",
        embedding_function=embeddings
    )

    print(f"Chunks in vectorstore: {vectorstore._collection.count()}")

In [ ]:
## Basic similarity search

`similarity_search()` takes a plain string, embeds it automatically, and returns the top-k most similar chunks.

`k` controls how many chunks come back — this is the **top-k** parameter. Higher k = more context, but also more noise. `k=3` is a reasonable default to start.

In [ ]:
query = "What does the moth look like?"
k = 3

if use_OpenAI:
    results = vectorstore.similarity_search(query, k=k)

    print(f"Query: '{query}'")
    print(f"Top {k} results:\n")

    for i, doc in enumerate(results):
        print(f"--- Result {i+1} ---")
        print(f"Source: {doc.metadata.get('source', 'unknown')}")
        print(doc.page_content)
        print()

## Try a different query

Swap in a query that should pull different chunks — this is a good way to build intuition for how semantic search behaves vs. keyword search.

In [ ]:
query2 = "Where is this species found geographically?"
k = 3

if use_OpenAI:
    results2 = vectorstore.similarity_search(query2, k=k)

    print(f"Query: '{query2}'")
    print(f"Top {k} results:\n")

    for i, doc in enumerate(results2):
        print(f"--- Result {i+1} ---")
        print(f"Source: {doc.metadata.get('source', 'unknown')}")
        print(doc.page_content)
        print()

## Wrap it in a reusable function

Once this is solid, we would move it to `src/helpers.py`
Then edit this notebook to import this, and apply it

In [ ]:
def retrieve(query, vectorstore, k=3):
    """
    Given a query string, return the top-k most relevant chunks from the vectorstore.
    Returns a list of LangChain Document objects.
    """
    results = vectorstore.similarity_search(query, k=k)
    return results


def print_results(query, results):
    print(f"Query: '{query}'\n")
    for i, doc in enumerate(results):
        print(f"--- Result {i+1} ---")
        print(f"Source: {doc.metadata.get('source', 'unknown')}")
        print(doc.page_content)
        print()




In [ ]:
# Test the functions
if use_OpenAI:
    test_query = "How does the moth camouflage itself?"
    test_results = retrieve(test_query, vectorstore, k=3)
    print_results(test_query, test_results)

## Summary

- `similarity_search(query, k=k)` is all it takes to retrieve relevant chunks
- The query is embedded on the fly using the same model as the stored chunks
- `k` is a key parameter — experiment with 3, 5, and 10 to see how result quality changes
- The `retrieve()` function above is ready to be moved to `src/helpers.py`

**Next (notebook 4):** pass these retrieved chunks as context to an LLM and generate an answer — the full RAG loop.